In [ ]:
import os
from autogen_core.models import UserMessage
from dotenv import load_dotenv
from autogen_ext.models.openai import OpenAIChatCompletionClient
from typing import List, Sequence
from autogen_agentchat.agents import AssistantAgent, UserProxyAgent
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.ui import Console
import requests
import xml.etree.ElementTree as ET

load_dotenv(override=True)

In [ ]:
gemini_key = os.getenv("GEMINI_API_KEY")

In [ ]:
model_client = OpenAIChatCompletionClient(
    model="gemini-3.1-flash-lite",
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=gemini_key,
    model_info={
        "vision": False,
        "function_calling": True,
        "json_output": False,
        "family": "unknown",
        "structured_output":True
    }
)

In [ ]:
if gemini_key:
    print(f"Success! API Key loaded (Length: {len(gemini_key)} characters)")
else:
    print("Error: API key not found")

In [ ]:
async def test_client():
    response = await model_client.create([
        UserMessage(content="Explain telepathy in one sentence", source="user")
    ])
    print(response.content)

await test_client()

In [ ]:
def arxiv_search_tool(query: str, max_results: int = 5) -> str:
    base_url = "http://export.arxiv.org/api/query"
    params = {
        "search_query" : query,
        "start" : 0,
        "max_results" : max_results
    }

    try:
        response = requests.get(base_url, params=params, timeout=10)

        if response.status_code != 200:
            return f"Error: Unable to get data from arxiv (status: {response.status_code})"

        root = ET.fromstring(response.content)

        namespaces = {"atom": "http://www.w3.org/2005/Atom"}
        entries = root.findall('atom:entry', namespaces)

        if not entries:
            return f"No papers found for the query: {query}"

        results = []
        for entry in entries:
            title_element = entry.find('atom:title', namespaces)
            abstract_element = entry.find('atom:summary', namespaces)
            link_element = entry.find('atom:id', namespaces)

            title = title_element.text.strip() if title_element is not None else "No Title Available"
            abstract = abstract_element.text.replace("\n", " ").strip() if abstract_element is not None else "No Abstract Available"
            link = link_element.text.strip() if link_element is not None else "No Link Available"

            result = f"Title: {title}\n Link: {link}\n Abstract: {abstract}\n"
            results.append(result)

        return "\n---\n".join(results)

    except Exception as e:
        return f"An error occurred while calling the arXiv API: {str(e)}"

In [ ]:
topic_refinement_agent = AssistantAgent(
    name = "topic_refinement_agent",
    model_client = model_client,
    description = "An agent that refines broad search ideas.",
    system_message =(
        """
        ROLE: You are an expert research topic refiner.
        TASK:
        1. Take a general research idea provided by the user.
        2. Analyze it and produce exactly one refined, precise academic research topic.

        Formatting Rules:
        - You must format your final output exactly as: Refined Research Topic: <Topic>
        - You must end your message with the exact termination text: TASK_COMPLETED_TOPIC_AGENT
        """
    )
)

In [ ]:
async def test_agent():
    await Console(topic_refinement_agent.run_stream(task="A broad research topic of your choice, centering on AI in environmental technology"))

In [ ]:
paper_discovery_agent = AssistantAgent(
    name = "paper_discovery_agent",
    model_client = model_client,
    tools = [arxiv_search_tool],
    description = "An agent that finds relevant academic papers on arXiv...",
    reflect_on_tool_use = False,
    system_message = (
        """
        ROLE: You are an expert academic paper discovery Agent.
        TASK: You MUST find ONLY 1 recent research paper related to the topic.

        OUTPUT FORMAT:
        ### Topic: <Topic>
        - Paper 1: <Title> - <Summary> - <URL>
        - You must end your final response with the exact termination text: TASK_COMPLETED_PAPER_AGENT
        """
    )
)

In [ ]:
insight_synthesizer_agent = AssistantAgent(
    name = "insight_synthesizer_agent",
    model_client = model_client,
    description = "An Agent that summarizes key insights from a discovered paper",
    system_message = """
        ROLE: You are an expert agent that synthesizes and summarizes key insights from the discovered paper
        TASK: You MUST provide 3 to 5 concise bullet points from the paper and you are to summarize common findings

        OUTPUT FORMAT:
        Key Insights:
        - <Insight 1>
        - <Insight 2>
        - <Insight 3>
        - You must end your final response with the exact termination text: TASK_COMPLETED_INSIGHT_AGENT
        """
)

In [ ]:
report_agent = AssistantAgent(
    name = "report_compiler_agent",
    model_client = model_client,
    description = "An Agent that compiles a concise research report",
    system_message = """
    ROLE: You are an expert report compiler
    TASK: You are to compile raw research insights into a readable report

    OUTPUT FORMAT: You MUST organize the compiled report in the following format
    Research Report
    1. Introduction
    2. Key Findings
    3. Discussion
    4. Conclusion

    At the very end of your final response, append the exact token: TASK_COMPLETED_REPORT_AGENT
    """
)

In [ ]:
gap_agent = AssistantAgent(
    name = "gap_analysis_agent",
    model_client = model_client,
    description = "An Agent that performs research gap analysis and proposes next step",
    system_message = """
    ROLE: You are an expert Research Gap Analyst Agent
    TASK: Your job is to analyse research reports and identify 1 research gaps and 1 future improvement direction

    OUTPUT FORMAT: You MUST organize your response in the following format:
    Research Gap:
    - <Research Gap>


    Future Improvement Direction:
    - <Future Improvement>


    At the very end of your final response, append the exact token: TASK_COMPLETED_GAP_AGENT
    """
)

In [ ]:
text_termination = TextMentionTermination(text="FINAL REPORT")

max_msg_termination = MaxMessageTermination(max_messages=10)

termination = text_termination | max_msg_termination


In [ ]:
user_proxy_agent = UserProxyAgent(
    name = "UserProxyAgent",
    description = "An agent for the user to review, approve or disapprove tasks"
)

In [ ]:
import sys
import time

user_approved = False

DOWNSTREAM_PIPELINE = {
    "paper_discovery_agent" : "insight_synthesizer_agent",
    "insight_synthesizer_agent" : "report_compiler_agent",
    "report_compiler_agent" : "gap_analysis_agent"
}

def selector_func_with_user_proxy(messages: Sequence[BaseAgentEvent | BaseChatMessage]) -> str | None :
    global user_approved

    if len(messages) == 0:
        return topic_refinement_agent.name

    last_source = messages[-1].source

    if not user_approved:
        if last_source == topic_refinement_agent.name:
            sys.stdout.flush()
            time.sleep(5.0)
            return user_proxy_agent.name

        if last_source == user_proxy_agent.name:
            content = messages[-1].content.upper()
            if "APPROVE" in content:
                user_approved = True
                return paper_discovery_agent.name

            else:
                return topic_refinement_agent.name

    else:
        if last_source in DOWNSTREAM_PIPELINE:
            return DOWNSTREAM_PIPELINE[last_source]

        if last_source == "gap_analysis_agent":
            return None

    return None


    # last_message = messages[-1]
    # last_speaker = last_message.source
    # last_content = last_message.content
    # text_content = last_content.upper() if isinstance(last_content, str) else ""
    #
    # if last_speaker not in [topic_refinement_agent.name, user_proxy_agent.name]:
    #     return topic_refinement_agent.name
    #
    # if last_speaker == topic_refinement_agent.name and not user_approved:
    #     sys.stdout.flush()
    #     time.sleep(0.5)
    #     return user_proxy_agent.name
    #
    # if last_speaker == user_proxy_agent.name:
    #     if "APPROVE" in text_content:
    #         user_approved = True
    #         return paper_discovery_agent.name
    #
    #     else:
    #         return topic_refinement_agent.name
    #
    # if user_approved:
    #     return None
    #
    # return None


In [ ]:
selector_prompt = """
You are the manager of a research assistant group chat.
Your job is to read the conversation history and select the next logical agent to proceed with the research.

The available agents and their roles are:
{roles}

Below is the conversation history:
{history}

Based on the history and roles, select the next agent to speak.
Return ONLY the name of the selected agents from the list: {participants}. Do not include any extra text.
"""

In [ ]:
task = "AI for Healthcare"

interactive_research_participants = [
        topic_refinement_agent,
        paper_discovery_agent,
        insight_synthesizer_agent,
        report_agent,
        gap_agent,
        user_proxy_agent
    ]

interactive_research_team = SelectorGroupChat(
    participants = interactive_research_participants,
    model_client = model_client,
    termination_condition = termination,
    selector_prompt = selector_prompt,
    selector_func = selector_func_with_user_proxy,
    allow_repeated_speaker = False
)

In [ ]:
await Console(interactive_research_team.run_stream(task=task))